In [1]:
import pandas as pd
import numpy as np
results = pd.read_csv('archive (1)/results.csv')
elo = pd.read_csv('eloratings.csv')
pastnames = pd.read_csv('archive (1)/former_names.csv')
shootouts = pd.read_csv('archive (1)/shootouts.csv')


result_teams = set(results["home_team"].unique()) | set(results["away_team"].unique())

uniqueTeams = set(results[results["date"] >= "2000-01-01"]["home_team"].unique()) | set(results[results["date"] >= "2000-01-01"]["away_team"].unique())
print(len(uniqueTeams))

shootouts["date"] = pd.to_datetime(shootouts["date"])
shootouts = shootouts.set_index(["date", "home_team"])



321


In [2]:
TeamNames = pd.unique(pd.concat([results["home_team"], results["away_team"]])).tolist()
Tournaments = pd.unique(results["tournament"]).tolist()

# lookup = pd.DataFrame({
#     "team": pd.Series(TeamNames),
#     "tournament": pd.Series(Tournaments)
# })

print(elo.dtypes)
print(elo.head())
# lookup.to_csv("lookup.csv", index=False)

date          str
team          str
rating    float64
change      int64
dtype: object
         date      team  rating  change
0  1872-11-30   England  2003.0       3
1  1872-11-30  Scotland  1997.0      -3
2  1873-03-08   England  2014.0      11
3  1873-03-08  Scotland  1986.0     -11
4  1874-03-07   England  2006.0      -8


In [3]:
elo["team"] = elo["team"].str.replace('\xa0', ' ', regex=False)

name_mapping = {
    "China": "China PR",
    "Czechia": "Czech Republic",
    "Ireland": "Republic of Ireland",
    "Macao": "Macau",
    "Macedonia": "North Macedonia",
    "East Germany": "German DR",
    "Democratic Republic of Congo": "DR Congo",
    "Congo-Brazzaville": "Congo",
    "East Timor": "Timor-Leste",
    "Reunion": "Réunion",
    "Sao Tome and Principe": "São Tomé and Príncipe",
    "US Virgin Islands": "United States Virgin Islands",
    "Wallis and Futuna": "Wallis Islands and Futuna",
    "Vatican": "Vatican City",
    "Soviet Union": "Russia",
    "West Germany": "Germany",
    "Saint Barthelemy": "Saint Barthélemy",
    "Khmer Republic": "Cambodia",
    "North Yemen": "Yemen",
    "South Vietnam": "Vietnam", 
    "Surinam": "Suriname",
    "Eastern Samoa": "American Samoa",
    "Federated States of Micronesia": "Micronesia",
}



elo["team"] = elo["team"].replace(name_mapping)

former_mapping = dict(zip(pastnames["former"], pastnames["current"]))
elo["team"] = elo["team"].replace(former_mapping)

elo_teams = set(elo["team"].unique())
result_teams = set(results["home_team"].unique()) | set(results["away_team"].unique())


only_in_elo = elo_teams - result_teams
only_in_results = result_teams - elo_teams



print("In Elo but not results:", sorted(only_in_elo))
print("In results but not Elo:", sorted(only_in_results))

elo["team"] = elo["team"].map({team: i for i, team in enumerate(TeamNames)}).fillna(-1).astype(int)

In Elo but not results: ['Christmas Island', 'Cocos Islands', 'Saba', 'Sint Eustatius']
In results but not Elo: ['Abkhazia', 'Alderney', 'Ambazonia', 'Andalusia', 'Arameans Suryoye', 'Artsakh', 'Asturias', 'Aymara', 'Barawa', 'Basque Country', 'Biafra', 'Brittany', 'Canary Islands', 'Cascadia', 'Catalonia', 'Central Spain', 'Chameria', 'Chechnya', 'Cilento', 'Corsica', 'County of Nice', 'Crimea', 'Darfur', 'Donetsk PR', 'Délvidék', 'East Turkestan', 'Elba Island', 'Ellan Vannin', 'Felvidék', 'Franconia', 'Frøya', 'Galicia', 'Gotland', 'Gozo', 'Guernsey', 'Găgăuzia', 'Hitra', 'Hmong', 'Iraqi Kurdistan', 'Isle of Man', 'Isle of Wight', 'Jersey', 'Kabylia', 'Kernow', 'Kárpátalja', 'Luhansk PR', 'Madrid', 'Manchukuo', 'Mapuche', 'Matabeleland', 'Maule Sur', 'Menorca', 'Occitania', 'Orkney', 'Padania', 'Panjab', 'Parishes of Jersey', 'Provence', 'Quebec', 'Raetia', 'Republic of St. Pauli', 'Rhodes', 'Romani people', 'Ryūkyū', 'Saare County', 'Saarland', 'Saint Helena', 'Sark', 'Saugeais', '

In [4]:
# Set up results
results = results.dropna(subset=["home_score", "away_score"]).reset_index(drop=True)

# Calculate days since last match for each team (vectorized)
results["date"] = pd.to_datetime(results["date"])
results_home = results[["date", "home_team"]].rename(columns={"home_team": "team"})
results_away = results[["date", "away_team"]].rename(columns={"away_team": "team"})
all_matches = pd.concat([results_home, results_away]).sort_values(["team", "date"]).reset_index(drop=True)
all_matches["prev_date"] = all_matches.groupby("team")["date"].shift(1)
all_matches["days_since_last"] = (all_matches["date"] - all_matches["prev_date"]).dt.days

home_days = all_matches.rename(columns={"team": "home_team", "days_since_last": "DaysSinceLastTeam1"})[["date", "home_team", "DaysSinceLastTeam1"]]
away_days = all_matches.rename(columns={"team": "away_team", "days_since_last": "DaysSinceLastTeam2"})[["date", "away_team", "DaysSinceLastTeam2"]]

results = results.merge(home_days, on=["date", "home_team"], how="left")
results = results.merge(away_days, on=["date", "away_team"], how="left")

results["DaysSinceLastTeam1"] = results["DaysSinceLastTeam1"].fillna(-1)
results["DaysSinceLastTeam2"] = results["DaysSinceLastTeam2"].fillna(-1)

dates = results["date"][5:]

cleaned = {"Date": dates.reset_index(drop=True),
           "Team1": [],
           "Team2": [],
           "Tournament": [],
           "Neutral": [],
           "DaysSinceLastTeam1": results["DaysSinceLastTeam1"][5:].reset_index(drop=True),
           "DaysSinceLastTeam2": results["DaysSinceLastTeam2"][5:].reset_index(drop=True),
           "h2hTeamOne": [],
           "h2hTeamTwo": [],
           "RFteamOne": [],
           "RFteamTwo": [],
           "RGFteamOne": [],
           "RGAteamOne": [],
           "RGFteamTwo": [],
           "RGAteamTwo": [],
           "shootoutWinner": [],
           "Winner": []}
TeamNames = pd.unique(pd.concat([results["home_team"], results["away_team"]])).tolist()
Tournaments = pd.unique(results["tournament"]).tolist()

for j in range(5, len(dates)+5):

    date = dates[j]

    resultsBeforeDate = results[:j]
    teamOne = results["home_team"][j]
    teamTwo = results["away_team"][j]

    teamOneGames = resultsBeforeDate[
        (resultsBeforeDate['home_team'] == teamOne) |
        (resultsBeforeDate['away_team'] == teamOne)].copy()
    teamOneGames['location'] = np.where(
        teamOneGames['home_team'] == teamOne, 'home', 'away')

    teamTwoGames = resultsBeforeDate[
        (resultsBeforeDate['home_team'] == teamTwo) |
        (resultsBeforeDate['away_team'] == teamTwo)].copy()
    teamTwoGames['location'] = np.where(
        teamTwoGames['home_team'] == teamTwo, 'home', 'away')

    h2hresults = resultsBeforeDate[resultsBeforeDate["date"] >= (pd.to_datetime(dates[j]) - pd.DateOffset(years=2)).strftime("%Y-%m-%d")].copy()
    h2hCount = len(h2hresults[
        ((h2hresults['home_team'] == teamOne) & (h2hresults['away_team'] == teamTwo)) |
        ((h2hresults['away_team'] == teamOne) & (h2hresults['home_team'] == teamTwo))].copy())

    winner = 0
    if results["home_score"][j] > results["away_score"][j]:
        winner = 1
    elif results["home_score"][j] < results["away_score"][j]:
        winner = 2

    teamOneDets = [0,0,0,0]  # gf, ga, rf, h2h
    teamTwoDets = [0,0,0,0]
    teamOneGamesPlayedWTeamTwo = 0
    teamTwoGamesPlayedWTeamOne = 0
    teamOneRecentGamesCount = 0
    teamTwoRecentGamesCount = 0

    for i in range(len(teamOneGames)-1, -1, -1):
        teamOneGame = teamOneGames.index[i]
        home_score = resultsBeforeDate["home_score"][teamOneGame]
        away_score = resultsBeforeDate["away_score"][teamOneGame]
        is_home = teamOneGames["location"].iloc[i] == "home"
        team_score = home_score if is_home else away_score
        opp_score = away_score if is_home else home_score
        opponent = (teamOneGames["away_team"].iloc[i] if is_home else teamOneGames["home_team"].iloc[i])

        if (len(teamOneGames)-5 <= i) if len(teamOneGames)-5 > 0 else True:
            if team_score > opp_score:
                teamOneDets[2] += 3
            elif team_score == opp_score:
                teamOneDets[2] += 1
            teamOneDets[0] += team_score
            teamOneDets[1] += opp_score
            teamOneRecentGamesCount += 1

        if opponent == teamTwo and teamOneGamesPlayedWTeamTwo < 5 and teamOneGamesPlayedWTeamTwo < h2hCount:
            if team_score > opp_score:
                teamOneDets[3] += 3
            elif team_score == opp_score:
                teamOneDets[3] += 1
            teamOneGamesPlayedWTeamTwo += 1

        if (teamOneGamesPlayedWTeamTwo >= h2hCount or teamOneGamesPlayedWTeamTwo >= 5) and teamOneRecentGamesCount > 4:
            break

    for i in range(len(teamTwoGames)-1, -1, -1):
        teamTwoGame = teamTwoGames.index[i]
        home_score = resultsBeforeDate["home_score"][teamTwoGame]
        away_score = resultsBeforeDate["away_score"][teamTwoGame]
        is_home = teamTwoGames["location"].iloc[i] == "home"
        team_score = home_score if is_home else away_score
        opp_score = away_score if is_home else home_score
        opponent = (teamTwoGames["away_team"].iloc[i] if is_home else teamTwoGames["home_team"].iloc[i])

        if (len(teamTwoGames)-5 <= i) if len(teamTwoGames)-5 > 0 else True:
            if team_score > opp_score:
                teamTwoDets[2] += 3
            elif team_score == opp_score:
                teamTwoDets[2] += 1
            teamTwoRecentGamesCount += 1
            teamTwoDets[0] += team_score
            teamTwoDets[1] += opp_score

        if opponent == teamOne and teamTwoGamesPlayedWTeamOne < 5 and teamTwoGamesPlayedWTeamOne < h2hCount:
            if team_score > opp_score:
                teamTwoDets[3] += 3
            elif team_score == opp_score:
                teamTwoDets[3] += 1
            teamTwoGamesPlayedWTeamOne += 1

        if (teamTwoGamesPlayedWTeamOne >= h2hCount or teamTwoGamesPlayedWTeamOne >= 5) and teamTwoRecentGamesCount > 4:
            break

    cleaned["Team1"].append(TeamNames.index(teamOne))
    cleaned["Team2"].append(TeamNames.index(teamTwo))
    cleaned["Tournament"].append(Tournaments.index(results["tournament"][j]))
    cleaned["Neutral"].append(int(results["neutral"][j]))

    cleaned["h2hTeamOne"].append(teamOneDets[3]/teamOneGamesPlayedWTeamTwo if teamOneGamesPlayedWTeamTwo > 0 else -1)
    cleaned["h2hTeamTwo"].append(teamTwoDets[3]/teamTwoGamesPlayedWTeamOne if teamTwoGamesPlayedWTeamOne > 0 else -1)
    cleaned["RFteamOne"].append(teamOneDets[2]/teamOneRecentGamesCount if teamOneRecentGamesCount > 0 else -1)
    cleaned["RFteamTwo"].append(teamTwoDets[2]/teamTwoRecentGamesCount if teamTwoRecentGamesCount > 0 else -1)
    cleaned["RGFteamOne"].append(teamOneDets[0]/teamOneRecentGamesCount if teamOneRecentGamesCount > 0 else 0)
    cleaned["RGAteamOne"].append(teamOneDets[1]/teamOneRecentGamesCount if teamOneRecentGamesCount > 0 else 0)
    cleaned["RGFteamTwo"].append(teamTwoDets[0]/teamTwoRecentGamesCount if teamTwoRecentGamesCount > 0 else 0)
    cleaned["RGAteamTwo"].append(teamTwoDets[1]/teamTwoRecentGamesCount if teamTwoRecentGamesCount > 0 else 0)
    cleaned["Winner"].append(winner)

    if winner == 0:
        shootout = shootouts["winner"].get((date, teamOne), "NONE")
        if shootout != "NONE":
            if shootout == teamOne:
                cleaned["shootoutWinner"].append(1)
            else:
                cleaned["shootoutWinner"].append(2)
        else:
            cleaned["shootoutWinner"].append(np.nan)
    else:
        cleaned["shootoutWinner"].append(winner)
    

    if j % 1000 == 0:
        print(j)

cleanedDF = pd.DataFrame(cleaned)

1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000


In [5]:
#set up elo
cleanedDF["Date"] = pd.to_datetime(cleanedDF["Date"])
elo["date"] = pd.to_datetime(elo["date"], format="mixed")

cleanedDF = cleanedDF.sort_values("Date").reset_index(drop=True)
elo = elo.sort_values("date").reset_index(drop=True)

# Merge Elo for Team1
elo_team1 = elo.rename(columns={"team": "Team1", "rating": "EloTeam1"})[["date", "Team1", "EloTeam1"]]
cleanedDF = pd.merge_asof(cleanedDF, elo_team1, left_on="Date", right_on="date", by="Team1")
cleanedDF = cleanedDF.drop(columns=["date"])

# Merge Elo for Team2
elo_team2 = elo.rename(columns={"team": "Team2", "rating": "EloTeam2"})[["date", "Team2", "EloTeam2"]]
cleanedDF = pd.merge_asof(cleanedDF, elo_team2, left_on="Date", right_on="date", by="Team2")
cleanedDF = cleanedDF.drop(columns=["date"])

# Drop rows where Elo is missing (the 155 unrated teams)
cleanedDF = cleanedDF.dropna(subset=["EloTeam1", "EloTeam2"])

print(cleanedDF.shape)
print(cleanedDF[["Date", "Team1", "EloTeam1", "Team2", "EloTeam2"]].head())



(43021, 19)
        Date  Team1  EloTeam1  Team2  EloTeam2
0 1876-03-25      0    2011.0      2    1499.0
1 1877-03-03      1    1990.0      0    2011.0
2 1877-03-05      2    1499.0      0    2011.0
3 1878-03-02      0    2011.0      1    1990.0
4 1878-03-23      0    2011.0      2    1499.0


In [6]:
np.random.seed(67)
mask = np.random.rand(len(cleanedDF)) < 0.5


cleanedDF.loc[mask, ["EloTeam1", "EloTeam2"]] = cleanedDF.loc[mask, ["EloTeam2", "EloTeam1"]].values

cleanedDF.loc[mask, ["Team1", "Team2"]] = cleanedDF.loc[mask, ["Team2", "Team1"]].values
cleanedDF.loc[mask, ["h2hTeamOne", "h2hTeamTwo"]] = cleanedDF.loc[mask, ["h2hTeamTwo", "h2hTeamOne"]].values

cleanedDF.loc[mask, ["RFteamOne", "RFteamTwo"]] = cleanedDF.loc[mask, ["RFteamTwo", "RFteamOne"]].values

cleanedDF.loc[mask, ["RGFteamOne", "RGFteamTwo"]] = cleanedDF.loc[mask, ["RGFteamTwo", "RGFteamOne"]].values
cleanedDF.loc[mask, ["RGAteamOne", "RGAteamTwo"]] = cleanedDF.loc[mask, ["RGAteamTwo", "RGAteamOne"]].values

cleanedDF.loc[mask, "Winner"] = np.where(cleanedDF.loc[mask, "Winner"] == 1, 2, 
                                         np.where(cleanedDF.loc[mask, "Winner"] == 2, 1, 0)
)




In [7]:


cleanedDF.to_csv('output.csv', index=False)    

# cleanedDF = pd.read_csv('output.csv',low_memory=False)    



In [ ]:
print(cleanedDF.shape)
print(cleanedDF.head())
print(cleanedDF["Winner"].value_counts())
print(cleanedDF.isnull().sum())

print(cleanedDF["Neutral"].value_counts())

In [8]:
lineups = pd.read_csv('players/game_lineups.csv', low_memory=False)
valuations = pd.read_csv('players/player_valuations.csv', low_memory=False)
players = pd.read_csv('players/players.csv', low_memory=False)
games = pd.read_csv('players/games.csv', low_memory=False)

# Join nationality onto valuations
squad_strength = valuations[["player_id", "date", "market_value_in_eur"]].merge(
    players[["player_id", "name", "country_of_citizenship"]], on="player_id", how="left"
)


# Convert date
squad_strength["date"] = pd.to_datetime(squad_strength["date"])
squad_strength["year"] = squad_strength["date"].dt.year

# For each country and date, average top 23 player values
squad_strength = (
    squad_strength.groupby(["country_of_citizenship", "date", "year"])
    .apply(lambda x: x.nlargest(23, "market_value_in_eur")["market_value_in_eur"].mean())
    .reset_index()
    .rename(columns={0: "squad_value"})
)

# Average across dates within each year to get one value per country per year
squad_yearly = squad_strength.groupby(["country_of_citizenship", "year"])["squad_value"].mean().reset_index()

# Normalise across countries within each year
squad_yearly["squad_value_normalized"] = squad_yearly.groupby("year")["squad_value"].transform(
    lambda x: (x - x.mean()) / x.std() if len(x) >= 10 else np.nan
)
squad_yearly["squad_value_normalized"] = squad_yearly["squad_value_normalized"].fillna(-1).clip(-3, 3)

print(squad_yearly["squad_value_normalized"].describe())

# Fix name mismatches
sq_name_mapping = {
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Brunei Darussalam": "Brunei",
    "China": "China PR",
    "Cote d'Ivoire": "Ivory Coast",
    "Curacao": "Curaçao",
    "Ireland": "Republic of Ireland",
    "Korea, North": "North Korea",
    "Korea, South": "South Korea",
    "Macao": "Macau",
    "Neukaledonien": "New Caledonia",
    "Saint-Martin": "Saint Martin",
    "Sao Tome and Principe": "São Tomé and Príncipe",
    "Southern Sudan": "South Sudan",
    "St. Kitts & Nevis": "Saint Kitts and Nevis",
    "St. Lucia": "Saint Lucia",
    "The Gambia": "Gambia",
    "Türkiye": "Turkey"
}
squad_yearly["country_of_citizenship"] = squad_yearly["country_of_citizenship"].replace(sq_name_mapping)

# Add a fake date column (Jan 1 of each year) for merge_asof
squad_yearly["date"] = pd.to_datetime(squad_yearly["year"].astype(str) + "-01-01")
squad_yearly = squad_yearly.sort_values("date").reset_index(drop=True)

# Add team name columns to cleanedDF
cleanedDF = cleanedDF.sort_values("Date").reset_index(drop=True)
cleanedDF["Team1_name"] = cleanedDF["Team1"].apply(lambda x: TeamNames[x] if x >= 0 else None)
cleanedDF["Team2_name"] = cleanedDF["Team2"].apply(lambda x: TeamNames[x] if x >= 0 else None)

# Check overlap
sq_teams = set(squad_yearly["country_of_citizenship"].unique())
cleaned_teams = set(cleanedDF["Team1_name"].dropna().unique())
print("Matching teams:", len(cleaned_teams.intersection(sq_teams)))

# Merge for Team1
sv_team1 = squad_yearly[["date", "country_of_citizenship", "squad_value_normalized"]].rename(
    columns={"country_of_citizenship": "Team1_name", "squad_value_normalized": "SquadValueTeam1"}
)
cleanedDF = pd.merge_asof(cleanedDF, sv_team1, left_on="Date", right_on="date", by="Team1_name")
cleanedDF = cleanedDF.drop(columns=["date"])

# Merge for Team2
sv_team2 = squad_yearly[["date", "country_of_citizenship", "squad_value_normalized"]].rename(
    columns={"country_of_citizenship": "Team2_name", "squad_value_normalized": "SquadValueTeam2"}
)
cleanedDF = pd.merge_asof(cleanedDF, sv_team2, left_on="Date", right_on="date", by="Team2_name")
cleanedDF = cleanedDF.drop(columns=["date"])

cleanedDF["SquadValueTeam1"] = cleanedDF["SquadValueTeam1"].fillna(-1)
cleanedDF["SquadValueTeam2"] = cleanedDF["SquadValueTeam2"].fillna(-1)

cleanedDF = cleanedDF.drop(columns=["Team1_name", "Team2_name"])

# Check result
post2013 = cleanedDF[cleanedDF["Date"] >= "2013-01-01"]
real_values = post2013[post2013["SquadValueTeam1"] != -1]["SquadValueTeam1"]
print(real_values.describe())
# real_values.hist(bins=50)
# plt.show()

count    3393.000000
mean       -0.018673
std         0.916136
min        -1.274805
25%        -0.715585
50%        -0.271807
75%         0.454208
max         3.000000
Name: squad_value_normalized, dtype: float64
Matching teams: 180
count    10595.000000
mean         0.054797
std          0.939198
min         -1.223920
25%         -0.680844
50%         -0.196858
75%          0.580427
max          3.000000
Name: SquadValueTeam1, dtype: float64


In [9]:
tournament_importance = {
    # Tier 5 - Major finals
    'FIFA World Cup': 5,
    'UEFA Euro': 5,
    'Copa América': 5,
    'African Cup of Nations': 5,
    'AFC Asian Cup': 5,
    'CONCACAF Championship': 5,
    'Gold Cup': 5,
    'Oceania Nations Cup': 5,
    'Olympic Games': 5,

    # Tier 4 - Secondary majors / confederations cups
    'Confederations Cup': 4,
    'UEFA Nations League': 4,
    'CONCACAF Nations League': 4,
    'CONMEBOL–UEFA Cup of Champions': 4,
    'Copa América qualification': 4,
    'AFC Challenge Cup': 4,
    'EAFF Championship': 4,
    'AFF Championship': 4,
    'COSAFA Cup': 4,
    'CECAFA Cup': 4,
    'Gulf Cup': 4,
    'WAFF Championship': 4,
    'SAFF Cup': 4,
    'CFU Caribbean Cup': 4,
    'CAFA Nations Cup': 4,
    'Superclásico de las Américas': 4,
    'Dynasty Cup': 4,
    'British Home Championship': 4,
    'Arab Cup': 4,
    'Asian Games': 4,
    'Pan American Championship': 4,
    'NAFC Championship': 4,
    'NAFU Championship': 4,
    'Melanesia Cup': 4,
    'Pacific Games': 4,
    'All-African Games': 4,
    'Bolivarian Games': 4,

    # Tier 3 - Qualifiers for major tournaments
    'FIFA World Cup qualification': 3,
    'UEFA Euro qualification': 3,
    'African Cup of Nations qualification': 3,
    'AFC Asian Cup qualification': 3,
    'CONCACAF Championship qualification': 3,
    'Gold Cup qualification': 3,
    'Oceania Nations Cup qualification': 3,
    'CONCACAF Nations League qualification': 3,
    'AFC Challenge Cup qualification': 3,
    'EAFF Championship qualification': 3,
    'AFF Championship qualification': 3,
    'ASEAN Championship qualification': 3,
    'COSAFA Cup qualification': 3,
    'CFU Caribbean Cup qualification': 3,
    'Arab Cup qualification': 3,
    'Copa América qualification': 3,
    'CONIFA World Cup qualification': 3,
    'CONIFA World Football Cup qualification': 3,

    # Tier 2 - Minor regional tournaments
    'ABCS Tournament': 2,
    'AFC Solidarity Cup': 2,
    'AFF Championship qualification': 2,
    'ASEAN Championship': 2,
    'African Friendship Games': 2,
    'Afro-Asian Games': 2,
    'Al Ain International Cup': 2,
    'Amílcar Cabral Cup': 2,
    'Atlantic Cup': 2,
    'Atlantic Heritage Cup': 2,
    'Balkan Cup': 2,
    'Baltic Cup': 2,
    'CCCF Championship': 2,
    'CONCACAF Series': 2,
    'CONIFA Africa Football Cup': 2,
    'CONIFA Asia Cup': 2,
    'CONIFA European Football Cup': 2,
    'CONIFA South America Football Cup': 2,
    'CONIFA World Football Cup': 2,
    'Central American and Caribbean Games': 2,
    'Central European International Cup': 2,
    'Copa Rio Branco': 2,
    'Copa Roca': 2,
    'Copa Oswaldo Cruz': 2,
    'Copa del Pacífico': 2,
    'ELF Cup': 2,
    'East Asian Games': 2,
    'Far Eastern Championship Games': 2,
    'FIFI Wild Cup': 2,
    'GaNEFo': 2,
    'Indian Ocean Island Games': 2,
    'Inter-Allied Games': 2,
    'Intercontinental Cup': 2,
    'Island Games': 2,
    'Kirin Cup': 2,
    'Kirin Challenge Cup': 2,
    'Lunar New Year Cup': 2,
    'Melanesia Cup': 2,
    'Merdeka Tournament': 2,
    'Mundialito': 2,
    'Muratti Vase': 2,
    'Nordic Championship': 2,
    'Oceania Nations Cup qualification': 2,
    'Pacific Mini Games': 2,
    'Rous Cup': 2,
    'SAFF Cup': 2,
    'South Asian Games': 2,
    'South Pacific Games': 2,
    'South Pacific Mini Games': 2,
    'Southeast Asian Games': 2,
    'Southeast Asian Peninsular Games': 2,
    'TIFOCO Tournament': 2,
    'UDEAC Cup': 2,
    'UNCAF Cup': 2,
    'UNIFFAC Cup': 2,
    'Viva World Cup': 2,
    'West African Cup': 2,
    'ConIFA Challenger Cup': 2,
    'Copa Artigas': 2,
    "Copa Bernardo O'Higgins": 2,
    'Copa Carlos Dittborn': 2,
    'Copa Chevallier Boutell': 2,
    'Copa Confraternidad': 2,
    'Copa Félix Bogado': 2,
    'Copa Juan Pinto Durán': 2,
    'Copa Lipton': 2,
    'Copa Newton': 2,
    'Copa Paz del Chaco': 2,
    'Copa Premio Honor Argentino': 2,
    'Copa Premio Honor Uruguayo': 2,
    'Copa Ramón Castilla': 2,
    'International Tournament of Peoples, Cultures and Tribes': 2,
    'The Other Final': 2,
    'Nile Basin Tournament': 2,
    'MSG Prime Minister\'s Cup': 2,

    # Tier 1 - Friendlies and invitational tournaments
    'Friendly': 1,
    'Beijing International Friendship Tournament': 1,
    'Benedikt Fontana Cup': 1,
    'Brazil Independence Cup': 1,
    'Canadian Shield': 1,
    'Corsica Cup': 1,
    "Coupe de l'Outre-Mer": 1,
    'Cup of Ancient Civilizations': 1,
    'Cyprus International Tournament': 1,
    'Dakar Tournament': 1,
    'Diamond Jubilee International Football Tournament': 1,
    'Dragon Cup': 1,
    'Dunhill Cup': 1,
    'EAFF Championship qualification': 1,
    'FIFA 75th Anniversary Cup': 1,
    'FIFA Series': 1,
    'Four Nations Tournament': 1,
    "Four Nations' Cup": 1,
    'Gold Cup qualification': 1,
    'Great Wall Cup': 1,
    'Guangzhou International Friendship Tournament': 1,
    'Hungary Heritage Cup': 1,
    'Indonesia Tournament': 1,
    'Inter Games': 1,
    'Joe Robbie Cup': 1,
    'Jordan International Tournament': 1,
    'King Hassan II Tournament': 1,
    "King's Cup": 1,
    'Korea Cup': 1,
    'Kuneitra Cup': 1,
    'Mahinda Rajapaksa Cup': 1,
    'Malta International Tournament': 1,
    'Mapinduzi Cup': 1,
    'Marianas Cup': 1,
    'Marlboro Cup': 1,
    'Matthews Cup': 1,
    'Mauritius Four Nations Cup': 1,
    'Merlion Cup': 1,
    'Miami Cup': 1,
    'Millennium Cup': 1,
    'Morocco, Capital of African Football': 1,
    'Mukuru 4 Nations': 1,
    'Navruz Cup': 1,
    'Nehru Cup': 1,
    'Niamh Challenge Cup': 1,
    'Nations Cup': 1,
    'OSN Cup': 1,
    'Open International Championship': 1,
    'Outrigger Challenge Cup': 1,
    'Pacific Mini Games': 1,
    'Palestine Cup': 1,
    'Palestine International Championship': 1,
    'Peace Cup': 1,
    'Philippine Peace Cup': 1,
    'Phillip Seaga Cup': 1,
    "Prime Minister's Cup": 1,
    'Real Madrid 75th Anniversary Cup': 1,
    'SKN Football Festival': 1,
    'Scania 100 Tournament': 1,
    'Simba Tournament': 1,
    'Soccer Ashes': 1,
    'South Asian Super Cup': 1,
    'Three Nations Cup': 1,
    'Tournament Burkina Faso': 1,
    'Tournoi de France': 1,
    'Trans-Tasman Cup': 1,
    'Tri Nation Tournament': 1,
    'Tri-Nations Series': 1,
    'Tynwald Hill Tournament': 1,
    'UNIFFAC Cup': 1,
    'USA Cup': 1,
    'United Arab Emirates Friendship Tournament': 1,
    'Unity Cup': 1,
    'VFF Cup': 1,
    'Vietnam Independence Cup': 1,
    'World Unity Cup': 1,
    'Zambian Independence Tournament': 1,
    'Évence Coppée Trophy': 1,
    'Windward Islands Tournament': 1,
    'NAFU Championship': 1,
    'Afro-Asian Games': 1,
    'Nile Basin Tournament': 1,
    'Inter-Allied Games': 1,
}
results["TournamentImportance"] = results["tournament"].map(tournament_importance).fillna(1).astype(int)
cleanedDF["TournamentImportance"] = results["tournament"].map(tournament_importance).fillna(1).astype(int)[5:].reset_index(drop=True)
cleanedDF = cleanedDF.drop(columns=["Tournament"])

In [10]:
cleanedDF["EloDiff"] = cleanedDF["EloTeam1"] - cleanedDF["EloTeam2"]
cleanedDF["SquadValueDiff"] = cleanedDF["SquadValueTeam1"] - cleanedDF["SquadValueTeam2"]

In [ ]:
trainingData = cleanedDF[(cleanedDF["Date"] >= "1900-01-01") & (cleanedDF["Date"] <= "2024-01-01")]
testingData = cleanedDF[cleanedDF["Date"] > "2024-01-01"]

X_train = trainingData.drop(columns=["Date", "Winner", "h2hTeamOne", "h2hTeamTwo", "TournamentImportance"])
X_test = testingData.drop(columns=["Date", "Winner", "h2hTeamOne", "h2hTeamTwo", "TournamentImportance"])

Y_train = trainingData["Winner"]
Y_test = testingData["Winner"]

filtered = cleanedDF[(cleanedDF["Date"] >= "2004-01-01")]

X = filtered.drop(columns=["Date", "Winner", "Neutral"])
Y = filtered["Winner"]


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

ai = RandomForestClassifier(class_weight="balanced")

ai.fit(X_train,Y_train)

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd


print(pd.Series(ai.feature_importances_, index=X_train.columns).sort_values(ascending=False))

print(classification_report(Y_test,ai.predict(X_test)))



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20, None],
    "min_samples_split": [2, 5, 10]
}

split = TimeSeriesSplit()
ai = RandomForestClassifier(class_weight="balanced",  n_jobs= 6)

search = GridSearchCV(ai, params, cv=split)
search.fit(X, Y)

print(search.best_params_)
print(search.best_score_)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV


split = TimeSeriesSplit()
xgb = XGBClassifier(eval_metric='mlogloss')

params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5,10,20],
    "learning_rate": [0.01,0.1,0.5]
}

#{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100} = 0.57 home 

#{'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300} 0.5740205027383795  home

#{'learning_rate': 0.012, 'max_depth': 5, 'n_estimators': 250} 0.5543322567055189 agumented 

#{'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 325} 0.5546692880213453


search = GridSearchCV(xgb, params, cv=split, n_jobs= 5)
search.fit(X, Y)

print(search.best_params_)
print(search.best_score_)



In [ ]:
print(pd.Series(search.best_estimator_.feature_importances_, index=X.columns).sort_values(ascending=False))


In [ ]:
post2013 = cleanedDF[cleanedDF["Date"] >= "2013-01-01"]
real_values = post2013[post2013["SquadValueTeam1"] != -1]["SquadValueTeam1"]
# print(real_values.describe())
# print(real_values.value_counts().head(20))

import matplotlib.pyplot as plt
real_values.hist(bins=50)
plt.show()


# print(squad_strength["squad_value_normalized"].describe())
# squad_strength["squad_value_normalized"].hist(bins=50)

# plt.show()


# print(squad_strength.groupby(["year"]).size().describe())
# print(squad_strength.groupby("year").size().sort_values())

print(squad_strength.groupby("year")["squad_value"].describe())

In [76]:
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight

split = TimeSeriesSplit()
xgb = XGBClassifier(eval_metric='mlogloss')

recentDF = cleanedDF[(cleanedDF["Date"] >= "2013-01-01") & (cleanedDF["Date"] <= "2024-01-01")]

params = {
    "n_estimators": [200, 225, 250],
    "max_depth": [3, 5, 10],
    "learning_rate": [0.05, 0.01, 0.1]
}

X = recentDF.drop(columns=["Date", "Winner", "h2hTeamOne", "h2hTeamTwo", "TournamentImportance", "shootoutWinner"])
Y = recentDF["Winner"]

GroupStageAI = GridSearchCV(xgb, params, cv=split, n_jobs=5)
GroupStageAI.fit(X, Y, sample_weight=compute_sample_weight('balanced', Y))


print(GroupStageAI.best_params_)
print(GroupStageAI.best_score_)
print(pd.Series(GroupStageAI.best_estimator_.feature_importances_, index=X.columns).sort_values(ascending=False))

{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 250}
0.5269354413161047
EloDiff               0.443780
EloTeam1              0.066710
RGAteamTwo            0.053851
SquadValueDiff        0.051715
RFteamTwo             0.047636
RGAteamOne            0.045146
Team2                 0.036857
RGFteamOne            0.033830
Team1                 0.032376
SquadValueTeam1       0.031838
EloTeam2              0.031743
RFteamOne             0.030276
DaysSinceLastTeam2    0.026344
SquadValueTeam2       0.025349
RGFteamTwo            0.021502
DaysSinceLastTeam1    0.021047
Neutral               0.000000
dtype: float32


In [80]:
testingData = cleanedDF[cleanedDF["Date"] > "2024-01-01"]

X_test = testingData.drop(columns=["Date", "Winner", "h2hTeamOne", "h2hTeamTwo", "TournamentImportance", "shootoutWinner"])

Y_test = testingData["Winner"]

proba = GroupStageAI.best_estimator_.predict_proba(X_test)

# Ignore draw probability, normalise between team1 and team2
knockout_preds = []
for p in proba:
    team1_prob = p[1] / (p[1] + p[2])
    knockout_preds.append(0 if team1_prob > 0.5 else 1)

# Remap Y_test to binary (drop draws)
binary_mask = Y_test != 0
binary_Y = Y_test[binary_mask].map({1: 0, 2: 1})
binary_preds = [knockout_preds[i] for i in range(len(Y_test)) if Y_test.iloc[i] != 0]

print(classification_report(binary_Y, binary_preds))

              precision    recall  f1-score   support

           0       0.78      0.79      0.78       911
           1       0.79      0.78      0.79       941

    accuracy                           0.79      1852
   macro avg       0.79      0.79      0.79      1852
weighted avg       0.79      0.79      0.79      1852



In [81]:
from sklearn.metrics import classification_report

testingData = cleanedDF[cleanedDF["Date"] > "2024-01-01"]

X_test = testingData.drop(columns=["Date", "Winner", "h2hTeamOne", "h2hTeamTwo", "TournamentImportance", "shootoutWinner"])

Y_test = testingData["Winner"]

proba = GroupStageAI.best_estimator_.predict_proba(X_test)
draw_threshold = 0.36

predictions = []
for p in proba:
    if p[0] > draw_threshold:
        predictions.append(0)
    else:
        predictions.append(1 if p[1] > p[2] else 2)

print(classification_report(Y_test, predictions))




              precision    recall  f1-score   support

           0       0.30      0.46      0.37       581
           1       0.69      0.56      0.62       911
           2       0.69      0.59      0.64       941

    accuracy                           0.55      2433
   macro avg       0.56      0.54      0.54      2433
weighted avg       0.60      0.55      0.57      2433

